In [2]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest
print("라이브러리 불러오기 완료")

라이브러리 불러오기 완료


In [3]:
import os

print(os.listdir("./data"))

['결제 정보 데이터.csv', '고객 센터 문의 데이터.csv', '구독 정보 데이터.csv', '유저 정보 데이터.csv', '일자별 행동 데이터.csv']


In [4]:
# 데이터 불러오기
pay = pd.read_csv("./data/결제 정보 데이터.csv")

# 각 데이터셋의 데이터 확인

## 결제 정보 데이터

In [5]:
pay.head()

,payment_date,user_id,subscription_type,amount,payment_status
0,2026-01-15,U2501,yearly,294,성공
1,2026-01-21,U2503,yearly,294,성공
2,2026-01-20,U2504,monthly,35,성공
3,2026-01-27,U2505,monthly,35,성공
4,2026-01-22,U2506,monthly,35,성공


In [6]:
pay.info()

<class 'pandas.DataFrame'>
RangeIndex: 940 entries, 0 to 939
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   payment_date       940 non-null    str  
 1   user_id            940 non-null    str  
 2   subscription_type  940 non-null    str  
 3   amount             940 non-null    int64
 4   payment_status     940 non-null    str  
dtypes: int64(1), str(4)
memory usage: 36.8 KB


In [7]:
pay.isna().sum()

payment_date         0
user_id              0
subscription_type    0
amount               0
payment_status       0
dtype: int64

In [8]:
pay.duplicated().sum()

np.int64(8)

In [9]:
pay[pay.duplicated()]

,payment_date,user_id,subscription_type,amount,payment_status
29,2026-01-17,U2590,monthly,35,실패
81,2026-01-19,U2745,monthly,35,실패
111,2026-01-17,U2831,monthly,35,실패
161,2026-01-27,U2961,monthly,35,실패
367,2026-01-21,U3473,yearly,294,실패
606,2026-01-18,U4104,yearly,294,실패
852,2026-01-18,U4759,monthly,35,실패
876,2026-01-17,U4823,yearly,294,실패


In [10]:
# 중복된 데이터만 모아서 보기
pay[pay.duplicated(keep=False)].sort_values(by='user_id')

,payment_date,user_id,subscription_type,amount,payment_status
28,2026-01-17,U2590,monthly,35,실패
29,2026-01-17,U2590,monthly,35,실패
80,2026-01-19,U2745,monthly,35,실패
81,2026-01-19,U2745,monthly,35,실패
110,2026-01-17,U2831,monthly,35,실패
111,2026-01-17,U2831,monthly,35,실패
160,2026-01-27,U2961,monthly,35,실패
161,2026-01-27,U2961,monthly,35,실패
366,2026-01-21,U3473,yearly,294,실패
367,2026-01-21,U3473,yearly,294,실패


In [11]:
# 완전히 똑같은 중복 행은 1건만 남기고 제거 (원본 데이터에 바로 반영)
pay.drop_duplicates(inplace=True)

# 제거 후 데이터 개수(행 수) 다시 확인
print("중복 제거 후 데이터 크기:", pay.shape)

중복 제거 후 데이터 크기: (932, 5)


In [12]:
pay.duplicated().sum()

np.int64(0)

In [13]:
# amount 컬럼의 요약 통계량 확인
pay['amount'].describe()

count    932.000000
mean     197.847639
std      125.199855
min       35.000000
25%       35.000000
50%      294.000000
75%      294.000000
max      294.000000
Name: amount, dtype: float64

- 940개 데이터중 8개의 데이터의 중복이 확인되어 확인 후 중복 제거를 진행
- 결측이나, 이상치로 보이는것은 없었다.

In [14]:
# payment_date 컬럼의 데이터 타입 확인
pay['payment_date'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [15]:
pay['payment_date'] = pd.to_datetime(pay['payment_date'])
print(pay['payment_date'].dtypes)

datetime64[us]


In [16]:
# 데이터의 가장 첫 날짜와 가장 마지막 날짜 확인
print("가장 빠른 결제일:", pay['payment_date'].min())
print("가장 늦은 결제일:", pay['payment_date'].max())

가장 빠른 결제일: 2026-01-15 00:00:00
가장 늦은 결제일: 2026-01-28 00:00:00


In [17]:
pay_success = pay[pay['payment_status'] == '성공']

print(pay_success['payment_status'].value_counts())

payment_status
성공    913
Name: count, dtype: int64


In [18]:
start_date = '2026-01-15'
end_date = '2026-01-28'

pay_final = pay_success[(pay_success['payment_date'] >= start_date) & (pay_success['payment_date'] <= end_date)]

print("필터링 후 시작일 : ", pay_final['payment_date'].min())
print("필터링 후 종료일 : ", pay_final['payment_date'].max())
print("최종 남은 데이터 건수 : ", pay_final.shape[0])

필터링 후 시작일 :  2026-01-15 00:00:00
필터링 후 종료일 :  2026-01-28 00:00:00
최종 남은 데이터 건수 :  913


In [19]:
pay_user_level = pay_final.groupby('user_id').agg(
    total_amount=('amount', 'sum'),
    monthly_cnt=('subscription_type', lambda x: (x == 'monthly').sum()),
    yearly_cnt=('subscription_type', lambda x: (x == 'yearly').sum())
).reset_index()

In [20]:
pay_user_level['is_purchased'] = 1

print(pay_user_level.head())

  user_id  total_amount  monthly_cnt  yearly_cnt  is_purchased
0   U2501           294            0           1             1
1   U2503           294            0           1             1
2   U2504            35            1           0             1
3   U2505            35            1           0             1
4   U2506            35            1           0             1


# 유저 정보 데이터

In [21]:
customer = pd.read_csv("./data/유저 정보 데이터.csv")

In [22]:
customer.head()

,user_id,user_type,group,gender,age
0,U2501,신규 고객,B,여성,24
1,U2502,신규 고객,A,여성,37
2,U2503,신규 고객,B,여성,30
3,U2504,신규 고객,A,남성,51
4,U2505,신규 고객,A,여성,37


## 결측치,중복값,이상치 확인

In [23]:
customer.isna().sum()

user_id      0
user_type    0
group        0
gender       0
age          0
dtype: int64

In [24]:
customer.duplicated().sum()

np.int64(0)

In [25]:
customer['group'].value_counts()

group
A    1277
B    1223
Name: count, dtype: int64

In [26]:
# 1. user_type 컬럼에 어떤 값들이 있는지 확인 (신규 고객만 있는지 검증)
print("유저 유형 분포:")
print(customer['user_type'].value_counts(dropna=False))
print("-" * 50)

# 2. gender(성별) 컬럼 고유값 확인
print("성별 분포:")
print(customer['gender'].value_counts(dropna=False))
print("-" * 50)

# 3. age(나이) 컬럼의 기초 통계량 확인 (말도 안 되는 나이가 있는지 검증)
print("나이 통계량:")
print(customer['age'].describe())

유저 유형 분포:
user_type
신규 고객    2500
Name: count, dtype: int64
--------------------------------------------------
성별 분포:
gender
남성    1271
여성    1229
Name: count, dtype: int64
--------------------------------------------------
나이 통계량:
count    2500.000000
mean       34.723600
std         9.076792
min        20.000000
25%        28.000000
50%        35.000000
75%        41.000000
max        60.000000
Name: age, dtype: float64


<> 유저 정보 데이터 끝

# 일자별 행동 데이터

In [27]:
dailybehavior = pd.read_csv("./data/일자별 행동 데이터.csv")

In [28]:
dailybehavior.head()

,user_id,date,device,visits,button_clicks_monthly,button_clicks_yearly
0,U2501,2026-01-15,PC,1,0,1
1,U2501,2026-01-16,PC,2,0,0
2,U2501,2026-01-17,PC,3,0,1
3,U2501,2026-01-18,PC,5,1,4
4,U2501,2026-01-20,PC,4,0,2


In [29]:
dailybehavior.isna().sum()

user_id                  0
date                     0
device                   0
visits                   0
button_clicks_monthly    0
button_clicks_yearly     0
dtype: int64

In [30]:
dailybehavior.duplicated().sum()

np.int64(0)

In [31]:
# 1. date 컬럼을 문자열에서 datetime(날짜) 타입으로 변환
dailybehavior['date'] = pd.to_datetime(dailybehavior['date'])

# 2. 행동 데이터의 시작일과 종료일 범위 확인
print("데이터 시작일:", dailybehavior['date'].min())
print("데이터 종료일:", dailybehavior['date'].max())
print("-" * 50)

# 3. 방문 수 및 버튼 클릭 수의 기초 통계량 확인 (말도 안 되는 수치가 있는지 검증)
print("[수치형 컬럼 통계량]")
print(dailybehavior[['visits', 'button_clicks_monthly', 'button_clicks_yearly']].describe())

데이터 시작일: 2026-01-15 00:00:00
데이터 종료일: 2026-01-28 00:00:00
--------------------------------------------------
[수치형 컬럼 통계량]
             visits  button_clicks_monthly  button_clicks_yearly
count  31484.000000           31484.000000          31484.000000
mean       2.991011               0.604402              0.898012
std        1.413972               0.854944              1.049932
min        1.000000               0.000000              0.000000
25%        2.000000               0.000000              0.000000
50%        3.000000               0.000000              1.000000
75%        4.000000               1.000000              1.000000
max        5.000000               5.000000              5.000000


In [32]:
# 1. 실험 기간 내 데이터만 필터링
start_date = '2026-01-15'
end_date = '2026-01-28'

df_behavior_filtered = dailybehavior[
    (dailybehavior['date'] >= start_date) & 
    (dailybehavior['date'] <= end_date)
]

# 2. user_id 기준으로 그룹화하여 총 방문수, 총 클릭수 집계
df_behavior_user_level = df_behavior_filtered.groupby('user_id').agg(
    total_visits=('visits', 'sum'),
    total_clicks_monthly=('button_clicks_monthly', 'sum'),
    total_clicks_yearly=('button_clicks_yearly', 'sum')
).reset_index()

# 3. 실험 기간 내에 방문한 이력이 있으므로 방문 여부 플래그를 1로 부여
df_behavior_user_level['is_visited'] = 1

# 최종 유저 단위 행동 데이터 확인
print(df_behavior_user_level.head())
print("실험 기간 내 방문한 고유 유저 수:", df_behavior_user_level.shape[0])

  user_id  total_visits  total_clicks_monthly  total_clicks_yearly  is_visited
0   U2501            39                     2                   20           1
1   U2502            45                    12                    6           1
2   U2503            39                     5                   18           1
3   U2504            40                    12                    9           1
4   U2505            27                     6                    2           1
실험 기간 내 방문한 고유 유저 수: 2500


<> 일자별 행동 데이터 끝

# 병합

In [33]:
# 1. 유저 정보 마스터에 행동 요약 데이터 합치기 (Left Join)
df_master = pd.merge(customer, df_behavior_user_level, on='user_id', how='left')

# 2. 위에서 합친 데이터에 결제 요약 데이터 데이터 또 합치기 (Left Join)
df_master = pd.merge(df_master, pay_user_level, on='user_id', how='left')

# 3. 빈 값(NaN) 채우기
# 행동 로그나 결제 기록이 없는 유저들은 Left Join 후 NaN으로 채워지므로, 이를 0으로 채워줍니다.
fill_values = {
    'total_visits': 0, 'total_clicks_monthly': 0, 'total_clicks_yearly': 0, 'is_visited': 0,
    'total_amount': 0, 'monthly_cnt': 0, 'yearly_cnt': 0, 'is_purchased': 0
}
df_master = df_master.fillna(value=fill_values)

# 4. 결과 검증 (전체 유저 수 2,500명이 그대로 유지되었는지 확인)
print("최종 마스터 테이블 크기:", df_master.shape)
df_master.head()

최종 마스터 테이블 크기: (2500, 13)


,user_id,user_type,group,gender,age,total_visits,total_clicks_monthly,total_clicks_yearly,is_visited,total_amount,monthly_cnt,yearly_cnt,is_purchased
0,U2501,신규 고객,B,여성,24,39,2,20,1,294.0,0.0,1.0,1.0
1,U2502,신규 고객,A,여성,37,45,12,6,1,0.0,0.0,0.0,0.0
2,U2503,신규 고객,B,여성,30,39,5,18,1,294.0,0.0,1.0,1.0
3,U2504,신규 고객,A,남성,51,40,12,9,1,35.0,1.0,0.0,1.0
4,U2505,신규 고객,A,여성,37,27,6,2,1,35.0,1.0,0.0,1.0


In [34]:
df_master.groupby('group')['total_amount'].agg(['count', 'mean']).reset_index()

,group,count,mean
0,A,1277,56.822240
1,B,1223,88.779231


In [35]:
# 1. A그룹과 B그룹의 매출 데이터 분리하기
A = df_master[df_master['group'] == 'A']['total_amount']
B = df_master[df_master['group'] == 'B']['total_amount']

In [36]:
lev_result = stats.levene(A, B)
print(f"p-value : {lev_result.pvalue : .3f}")

p-value :  0.000


- p-value < 유의수준(0.05) : 귀무가설 기각. 등분산성 불만족.

In [37]:
# 웰치 t-검정
t_result = stats.ttest_ind(A, B, equal_var=False)

print(f"A그룹의 평균 : {A.mean() : ,.0f}, B그룹의 평균 : {B.mean(): ,.0f}")
print(f"p-value : {t_result.pvalue : .3f}")

A그룹의 평균 :  57, B그룹의 평균 :  89
p-value :  0.000


- p-value < 유의수준 : 귀무가설 기각.


## 보조 지표

### 월간 멤버십 구독 전환율

In [38]:
# 1. 피벗 테이블 생성
pivot_df = pd.pivot_table(df_master,
                          index='group',
                          columns='monthly_cnt',
                          values='user_id',
                          aggfunc='count')

# 2. 명칭 변경 및 인덱스 정돈
pivot_df.columns = ['미구독', '구독']
pivot_df.index.name = 'group'

# 3. 전체 및 전환율 계산
pivot_df['전체'] = pivot_df['구독'] + pivot_df['미구독']
pivot_df['전환율(%)'] = (pivot_df['구독'] / pivot_df['전체']) * 100

print("=== 월간 멤버십 구독 현황 ===")
display(pivot_df.round(2)) 
print("-" * 60)

# 4. 모비율 Z-검정 데이터 추출
count = pivot_df['구독']
nobs = pivot_df['전체']
z_stat, p_value = proportions_ztest(count, nobs)

# 5. 최종 결과 출력 (대소문자 자동 맞춰서 출력)
group_a = 'A' if 'A' in pivot_df.index else 'a'
group_b = 'B' if 'B' in pivot_df.index else 'b'

print(f"{group_a}그룹 월간 구독 전환율 : {pivot_df.loc[group_a, '전환율(%)']:.2f}%, {group_b}그룹 월간 구독 전환율 : {pivot_df.loc[group_b, '전환율(%)']:.2f}%")
print(f"p-value : {p_value:.3f}")

=== 월간 멤버십 구독 현황 ===


,미구독,구독,전체,전환율(%)
group,,,,
A,1035,242,1277,18.95
B,1128,95,1223,7.77


------------------------------------------------------------
A그룹 월간 구독 전환율 : 18.95%, B그룹 월간 구독 전환율 : 7.77%
p-value : 0.000


- p-value < 유의수준 : 귀무가설 기각. 두 그룹의 구독 전환율에는 통계적으로 유의미한 차이가 있다고 할만한 충분한 근거가 있으나, A그룹의 구독 전환율이 B그룹의 구독 전환율보다 높아서, 변경안이 기대만큼의 성과를 내지 못했다고 해석할 수 있다.

### 연간 멤버십 구독 전환율

In [39]:
# 1. 피벗 테이블 생성
pivot_df = pd.pivot_table(df_master,
                          index='group',
                          columns='yearly_cnt',
                          values='user_id',
                          aggfunc='count')

# 2. 명칭 변경 및 인덱스 정돈
pivot_df.columns = ['미구독', '구독']
pivot_df.index.name = 'group'

# 3. 전체 및 전환율 계산
pivot_df['전체'] = pivot_df['구독'] + pivot_df['미구독']
pivot_df['전환율(%)'] = (pivot_df['구독'] / pivot_df['전체']) * 100

print("=== 연간 멤버십 구독 현황 ===")
display(pivot_df.round(2)) 
print("-" * 60)

# 4. 모비율 Z-검정 데이터 추출
count = pivot_df['구독']
nobs = pivot_df['전체']
z_stat, p_value = proportions_ztest(count, nobs)

# 5. 최종 결과 출력 (대소문자 자동 맞춰서 출력)
group_a = 'A' if 'A' in pivot_df.index else 'a'
group_b = 'B' if 'B' in pivot_df.index else 'b'

print(f"{group_a}그룹 연간 구독 전환율 : {pivot_df.loc[group_a, '전환율(%)']:.2f}%, {group_b}그룹 연간 구독 전환율 : {pivot_df.loc[group_b, '전환율(%)']:.2f}%")
print(f"p-value : {p_value:.3f}")

=== 연간 멤버십 구독 현황 ===


,미구독,구독,전체,전환율(%)
group,,,,
A,1059,218,1277,17.07
B,865,358,1223,29.27


------------------------------------------------------------
A그룹 연간 구독 전환율 : 17.07%, B그룹 연간 구독 전환율 : 29.27%
p-value : 0.000


- p-value < 유의수준 : 귀무가설 기각. A그룹의 연간 구독 전환율은 17.07% B그룹의 연간 구독 전환율은 29.27% 약 12.20% 수준 상승한것으로 보아 B안의 변경안이 기대만큼의 성과를 냈다 볼 수 있다.

# 가드레일 지표

In [41]:
df_cs = pd.read_csv("./data/고객 센터 문의 데이터.csv")

In [42]:
df_cs.head()

,date,user_id,inquiry_type
0,2026-01-27,U2506,결제/영수증 문의
1,2026-01-20,U2525,결제/영수증 문의
2,2026-01-12,U2552,서비스 이용 문의
3,2026-01-22,U2554,환불 문의
4,2026-01-25,U2572,서비스 이용 문의


In [45]:
refund_users = df_cs[df_cs['inquiry_type'] == '환불 문의']['user_id'].unique()
refund_users

<StringArray>
['U2554', 'U2590', 'U2660', 'U2666', 'U2721', 'U2729', 'U2947', 'U3019',
 'U3193', 'U3305', 'U3323', 'U3418', 'U3445', 'U3493', 'U3535', 'U3625',
 'U3871', 'U4006', 'U4022', 'U4099', 'U4309', 'U4354', 'U4450', 'U4590',
 'U4643', 'U4653', 'U4665', 'U4687', 'U4709', 'U4718', 'U4834', 'U4987']
Length: 32, dtype: str

In [46]:
df_master['환불_문의여부'] = df_master['user_id'].isin(refund_users).astype(int)
df_master['환불_문의여부']

0       0
1       0
2       0
3       0
4       0
       ..
2495    0
2496    0
2497    0
2498    0
2499    0
Name: 환불_문의여부, Length: 2500, dtype: int64

In [47]:
pivot_refund = pd.pivot_table(df_master,
                              index='group',
                              columns='환불_문의여부',
                              values='user_id',
                              aggfunc='count')

pivot_refund.columns = ['미문의', '문의']
pivot_refund.index.name = 'group'

pivot_refund['전체'] = pivot_refund['문의'] + pivot_refund['미문의']
pivot_refund['문의율'] = pivot_refund['문의'] / pivot_refund['전체']

In [ ]:
print("=== 환불 문의 현황 ===")
display(pivot_refund)
print("-" * 60)

count_ref = pivot_refund['문의']
nobs_ref = pivot_refund['전체']

z_stat_ref, p_value_ref = proportions_ztest(count_ref, nobs_ref)

print(f"{group_a}그룹 환불 문의율 : {pivot_refund.loc[group_a, '문의율']:.4f}, {group_b}그룹 환불 문의율 : {pivot_refund.loc[group_b, '문의율']:.4f}")
print(f"p-value : {p_value_ref:.3f}")

=== 환불 문의 현황 ===


,미문의,문의,전체,문의율
group,,,,
A,1267,10,1277,0.007831
B,1201,22,1223,0.017989


------------------------------------------------------------
A그룹 환불 문의율 : 0.0078, B그룹 환불 문의율 : 0.0180
p-value : 0.024


- p-value < 유의수준 (0.05): 귀무가설 기각. A그룹의 환불 문의율은 0.78%, B그룹은 1.80%로 약 1.02% 수준 유의미하게 상승함. B안(변경안)이 연간 구독률을 높였으나 부작용으로 환불 문의 또한 함께 증가시켰으므로 가드레일 지표 측면에서 추가적인 원인 분석(유저 불편 요소 점검 등)이 필요하다.

In [49]:
df_sub = pd.read_csv("./data/구독 정보 데이터.csv")

In [50]:
df_sub.head()

,user_id,subscription_type,subscribed_date,unsubscribed_date
0,U2501,yearly,2026-01-15,NaN
1,U2503,yearly,2026-01-21,NaN
2,U2504,monthly,2026-01-20,2026-01-22
3,U2505,monthly,2026-01-27,NaN
4,U2506,monthly,2026-01-22,NaN


In [51]:
df_sub['subscribed_date'] = pd.to_datetime(df_sub['subscribed_date'])
df_sub['unsubscribed_date'] = pd.to_datetime(df_sub['unsubscribed_date'])

In [53]:
start_date = '2026-01-15'
end_date = '2026-01-28'

df_sub_filtered = df_sub[(df_sub['subscribed_date'] >= start_date) & 
                         (df_sub['subscribed_date'] <= end_date)].copy()

df_sub_filtered['실제_취소여부'] = df_sub_filtered['unsubscribed_date'].notnull().astype(int)

In [56]:
df_sub_to_merge = df_sub_filtered[['user_id', 'subscription_type', '실제_취소여부']]
df_analysis = pd.merge(df_master, df_sub_to_merge, on='user_id', how='left')

df_analysis['실제_취소여부'] = df_analysis['실제_취소여부'].fillna(0).astype(int)

In [58]:
for sub_type in ['monthly', 'yearly']:
    # 해당 구독 유형인 유저들만 필터링
    df_target = df_analysis[df_analysis['subscription_type'] == sub_type]
    
    pivot_sub = pd.pivot_table(df_target,
                               index='group',
                               columns='실제_취소여부',
                               values='user_id',
                               aggfunc='count').fillna(0)
    
    # 컬럼 및 인덱스 명칭 정돈 (0 -> 유지, 1 -> 취소)
    pivot_sub.columns = ['유지', '취소']
    pivot_sub.index.name = 'group'
    
    pivot_sub['전체'] = pivot_sub['취소'] + pivot_sub['유지']
    pivot_sub['취소율'] = pivot_sub['취소'] / pivot_sub['전체']
    
    type_kor = "월간" if sub_type == 'monthly' else "연간"
    print(f"=== {type_kor} 실제 구독 취소 현황 ===")
    display(pivot_sub)
    
    # 모비율 Z-검정 데이터 추출
    count_sub = pivot_sub['취소']
    nobs_sub = pivot_sub['전체']
    
    if len(count_sub) == 2 and (count_sub > 0).all():
        _, p_value_sub = proportions_ztest(count_sub, nobs_sub)
        
        group_a = 'A' if 'A' in pivot_sub.index else 'a'
        group_b = 'B' if 'B' in pivot_sub.index else 'b'
        
        print(f"{group_a}그룹 {type_kor} 취소율 : {pivot_sub.loc[group_a, '취소율']:.4f}, {group_b}그룹 {type_kor} 취소율 : {pivot_sub.loc[group_b, '취소율']:.4f}")
        print(f"p-value : {p_value_sub:.3f}")
    else:
        print("데이터 부족 혹은 취소 건수 없음으로 Z-test 생략")
    print("-" * 60)

=== 월간 실제 구독 취소 현황 ===


,유지,취소,전체,취소율
group,,,,
A,205,37,242,0.152893
B,82,13,95,0.136842


A그룹 월간 취소율 : 0.1529, B그룹 월간 취소율 : 0.1368
p-value : 0.709
------------------------------------------------------------
=== 연간 실제 구독 취소 현황 ===


,유지,취소,전체,취소율
group,,,,
A,181,37,218,0.169725
B,293,65,358,0.181564


A그룹 연간 취소율 : 0.1697, B그룹 연간 취소율 : 0.1816
p-value : 0.718
------------------------------------------------------------


* **월간 멤버십 실제 취소율 (p-value: 0.709):** 귀무가설 채택. A그룹(15.29%)과 B그룹(13.68%) 간의 취소율 차이는 통계적으로 유의미하지 않음.
* **연간 멤버십 실제 취소율 (p-value: 0.718):** 귀무가설 채택. A그룹(16.97%)과 B그룹(18.16%) 간의 취소율 차이는 통계적으로 유의미하지 않음.
* **해석 및 결론:** B안(변경안) 적용 시 단가가 높은 연간 구독률이 폭발적으로 증가했음에도 불구하고, 이후 유저들의 실제 구독 취소(이탈) 행태는 기존 안과 다름없이 안전하게 유지됨을 확인하여 가드레일 지표를 성공적으로 통과함.